# Packages

In [ ]:
!pip install "jax==0.7.2" "jaxlib==0.7.2" "diffrax==0.7.0" "equinox" "optax"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from jax import jit
import functools
from time import time

jax.config.update("jax_enable_x64", True)

# ---------- Colab-friendly import of shared Dynamics/Training modules ----------
import os, sys

REPO_URL = "https://github.com/YanchengDu/LiquidCHL.git"
REPO_DIR = "LiquidCHL"

if os.path.isdir("Dynamics") and os.path.isdir("Training"):
    repo_root = "."
elif os.path.isdir(os.path.join("..", "Dynamics")):
    repo_root = ".."
else:
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    repo_root = REPO_DIR

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from Dynamics.Spatial_Cahn_Hilliard import (
    simplex_project_np, simplex_project_jax,
    init_phi_multi_jax,
    fprime_multi_jax,
    compute_energy_spectral_jax, compute_energy_map_jax,
    step_jax,
    run_cahn_hilliard_multi_FH_rstab_jax_until_converged,
    partition_memory, hebbian_learning_matrix,
    valid_vectors, generate_vectors, generate_memories,
    identify_memories,
    plot_energy, plot_final_components, plot_memories_grayscale,
)


# Target memories

In [4]:
N = 32
target_memories = ([[1.,0.,0.,0.,0.,0.,1.,1.,0.,0.,1.,0.,1.,1.,0.,1.,0.,1.,0.,0.,0.,0.,1.,1.,1.,1.,0.,1.,0.,1.,1.,1.],
 [1.,0.,0.,0.,1.,0.,0.,0.,1.,0.,1.,0.,1.,0.,0.,1.,1.,1.,0.,0.,1.,1.,0.,1.,1.,0.,1.,0.,0.,1.,1.,1.],
 [0.,1.,1.,1.,1.,1.,0.,0.,0.,1.,1.,0.,0.,1.,0.,1.,0.,1.,1.,0.,1.,1.,0.,0.,1.,0.,0.,0.,0.,1.,1.,0.],
 [0.,0.,0.,1.,0.,0.,0.,0.,0.,1.,0.,0.,1.,1.,1.,1.,0.,1.,1.,1.,1.,1.,1.,0.,1.,0.,1.,1.,0.,1.,0.,0.],
 [1.,1.,1.,0.,0.,1.,0.,1.,0.,0.,0.,1.,0.,0.,0.,1.,0.,1.,0.,1.,0.,0.,1.,0.,1.,1.,1.,0.,1.,1.,0.,1.]])
partitioned_memories = []
beta = 0.5
for memory in target_memories:
    partitioned_memories.append(partition_memory(memory, beta))
print("Partitioned memories:")
for idx, p_memory in enumerate(partitioned_memories):
    print(f"Memory {idx+1}: {p_memory}")
# Generate initial condition
memory_ratio = [1.0/len(partitioned_memories) for _ in range(len(partitioned_memories))]
mean_phi = np.mean(target_memories)
chi_ref = []
chi_ref.append(hebbian_learning_matrix(partitioned_memories, memory_ratio))
# Plot the reference chi matrix
plt.figure(figsize=(6, 5))
plt.imshow(chi_ref[0], cmap='coolwarm', vmin=-np.max(np.abs(chi_ref[0])), vmax=np.max(np.abs(chi_ref[0])))
plt.colorbar(label='Interaction Strength (chi)')
plt.title('Reference Interaction Matrix (chi)')
plt.tight_layout()
plt.show()

# Import trained matrix

In [ ]:
# Download trained interaction matrix from GitHub
import urllib.request

RAW_URL = "https://raw.githubusercontent.com/YanchengDu/LiquidCHL/main/Spatial_Phase/chi_trained.npy"
urllib.request.urlretrieve(RAW_URL, "chi_trained.npy")
print("Downloaded chi_trained.npy")


In [6]:
chi_trained = np.load("chi_trained.npy")

# Initialize

In [7]:
nc = 32
nx=ny=128
phi_initial = init_phi_multi_jax(
            nc=nc,
            nx=nx,
            ny=ny,
            mean_phi=jnp.mean(np.array(partitioned_memories),axis=0),
            noise_amp=0.003,
            seed=42,
            sigma_spatial=2.0,
        )

# Running 

In [9]:
nc=32
frames,phi,mean_phi_global,energies,steps_saved,time_saved = run_cahn_hilliard_multi_FH_rstab_jax_until_converged(
    nc=32,
    nx=128,
    ny=128,
    dx=1.0,
    dy=1.0,
    M=jnp.ones(shape=nc),
    kappa=5.0*jnp.ones(shape=nc),
    chi=chi_trained,
    nu=None,
    dt=1e-3,
    mean_phi=None,
    initial_phi=phi_initial,
    noise_amp=0.02,
    save_interval=1000,
    seed=None,
    verbose=True,
    track_energy=True,
    training=False,
    r_stab=None,
    max_steps=200000,
    max_time=500.0,
    dt_min=1e-8,
    dt_max=0.1,
    energy_increase_tol=1e-6,
    mean_tol=1e-5,
    raw_mass_tol=1e-5,
    raw_min_tol=-1e-6,
    field_tol=1e-6,
    energy_slope_tol=1e-8,
    window=5,
    min_steps_check=5,
    grow_dt=1.05,
    sigma_spatial=2.0,
)
plot_energy(time_saved, energies, title=f"Energy decay")

In [13]:
# ======================
# Inputs:
# closest_indices       : int array (128,128), values 0...K-1
# min_distances         : float array (128,128)
# FH_total_energy_map   : float array (128,128)
# ======================

nx=ny=128
nc=32
chi_trained = np.load('chi_trained.npy')
kx = 2.0 * np.pi * np.fft.fftfreq(nx, d=1.0)
ky = 2.0 * np.pi * np.fft.fftfreq(ny, d=1.0)
K2 = kx[:, None] ** 2 + ky[None, :] ** 2
filt = np.exp(-0.5 * (2.0 ** 2) * K2)
print(frames.shape)
FH_local_energy_map, FH_grad_energy_map, FH_total_energy_map = compute_energy_map_jax(frames[-1], chi_trained, kappa=5.0*jnp.ones(shape=nc),K2=K2)
average_vectors, figure, closest_indices, min_distances=identify_memories(frames[-1].transpose(1,2,0), partitioned_memories)
K = 5
desired_indices = [0, 4, 8, 10, 6]

# ----------------------
# Colormap for memory map
# ----------------------
base_colors = plt.cm.get_cmap("tab20")
custom_colors = [base_colors.colors[i] for i in desired_indices]
cmap_custom = ListedColormap(custom_colors)

# ----------------------
# Convert distances to alpha
# ----------------------
d = min_distances.astype(float)

d_min = d.min()
d_max = 0.30

d_norm = (d - d_min) / (d_max - d_min + 1e-12)
d_norm = np.clip(d_norm, 0, 1)

# small distance -> opaque
alpha = 1.0 - d_norm
alpha = 0.15 + 0.85 * alpha

print("distance min/max:", d.min(), d.max())

# ----------------------
# Construct RGBA image
# ----------------------
rgba_img = cmap_custom(closest_indices.astype(int))
rgba_img[..., 3] = alpha

# ----------------------
# Energy color limits
# ----------------------
vmin = np.min(FH_total_energy_map)
vmax = np.max(FH_total_energy_map)

# ======================
# Side-by-side plot
# ======================
fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 5),
    gridspec_kw={"width_ratios": [1, 1]}
)

ax0, ax1 = axes

# ----------------------
# Left: closest memory map
# ----------------------
ax0.imshow(rgba_img, interpolation="nearest")
ax0.set_xticks([])
ax0.set_yticks([])
ax0.invert_yaxis()
ax0.set_title("Closest Memory")

# ----------------------
# Right: total energy map
# ----------------------
im = ax1.imshow(
    FH_total_energy_map,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest"
)
ax1.set_xticks([])
ax1.set_yticks([])
ax1.invert_yaxis()
ax1.set_title("Total Energy")

cbar = fig.colorbar(
    im,
    ax=ax1,
    fraction=0.046,
    pad=0.04
)
cbar.set_label("Total Energy")

# Make room on the far right for alpha mini legends
fig.subplots_adjust(right=0.84, wspace=0.15)

# Need finalized axis positions
fig.canvas.draw()
pos = ax0.get_position()

# ======================
# Vertical alpha legends for left plot
# ======================
legend_gap = 0.008
legend_width = 0.018
legend_x_offset = 0.015

total_height = pos.height
box_height = (total_height - (K - 1) * legend_gap) / K

n_alpha = 100

for k in range(K):

    bottom = pos.y0 + (K - 1 - k) * (box_height + legend_gap)

    ax_leg = fig.add_axes([
        pos.x1 + legend_x_offset,
        bottom,
        legend_width,
        box_height
    ])

    color = np.array(cmap_custom(k))

    bar = np.ones((n_alpha, 1, 4))
    bar[:, :, :3] = color[:3]
    bar[:, :, 3] = np.linspace(0.15, 1.0, n_alpha)[:, None]

    ax_leg.imshow(
        bar,
        origin="lower",
        aspect="auto",
        interpolation="nearest"
    )

    ax_leg.set_xticks([])
    ax_leg.set_yticks([])

    ax_leg.text(
        1.7,
        0.5,
        f"{k + 1}",
        transform=ax_leg.transAxes,
        ha="left",
        va="center",
        fontsize=9
    )

    for spine in ax_leg.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

plt.show()

In [15]:
frames_concat = frames

In [36]:
frames_new,phi,mean_phi_global,energies,steps_saved,time_saved = run_cahn_hilliard_multi_FH_rstab_jax_until_converged(
    nc=32,
    nx=128,
    ny=128,
    dx=1.0,
    dy=1.0,
    M=jnp.ones(shape=nc),
    kappa=5.0*jnp.ones(shape=nc),
    chi=chi_trained,
    nu=None,
    dt=1e-3,
    mean_phi=None,
    initial_phi=frames_concat[-1],
    noise_amp=0.02,
    save_interval=1000,
    seed=None,
    verbose=True,
    track_energy=True,
    training=False,
    r_stab=None,
    max_steps=200000,
    max_time=500.0,
    dt_min=1e-8,
    dt_max=0.1,
    energy_increase_tol=1e-6,
    mean_tol=1e-5,
    raw_mass_tol=1e-5,
    raw_min_tol=-1e-6,
    field_tol=1e-6,
    energy_slope_tol=1e-8,
    window=5,
    min_steps_check=5,
    grow_dt=1.05,
    sigma_spatial=2.0,
)

In [194]:
# ======================
# Inputs / calculation
# ======================
print(frames_concat.shape)
print(frames_new.shape)

frames_concat = np.concatenate((frames_concat, frames_new), axis=0)

print(frames_concat.shape)

nx = ny = 128
nc = 32

chi_trained = np.load("chi_trained.npy")

kx = 2.0 * np.pi * np.fft.fftfreq(nx, d=1.0)
ky = 2.0 * np.pi * np.fft.fftfreq(ny, d=1.0)
K2 = kx[:, None] ** 2 + ky[None, :] ** 2

filt = np.exp(-0.5 * (2.0 ** 2) * K2)

FH_local_energy_map, FH_grad_energy_map, FH_total_energy_map = compute_energy_map_jax(
    frames_concat[-1],
    chi_trained,
    kappa=5.0 * jnp.ones(shape=nc),
    K2=K2
)

average_vectors, figure, closest_indices, min_distances = identify_memories(
    frames_concat[-1].transpose(1, 2, 0),
    partitioned_memories
)

# Convert JAX arrays to NumPy if needed
FH_total_energy_map = np.asarray(FH_total_energy_map)
closest_indices = np.asarray(closest_indices)
min_distances = np.asarray(min_distances)

# ======================
# Colormap
# ======================
K = 5
desired_indices = [0, 4, 8, 10, 6]

base_colors = plt.cm.get_cmap("tab20")
custom_colors = [base_colors.colors[i] for i in desired_indices]
cmap_custom = ListedColormap(custom_colors)

# ======================
# Alpha from distance
# ======================
d = min_distances.astype(float)

d_min = d.min()
d_max = 0.30

d_norm = (d - d_min) / (d_max - d_min + 1e-12)
d_norm = np.clip(d_norm, 0, 1)

alpha = 1.0 - d_norm
alpha = 0.15 + 0.85 * alpha

print("distance min/max:", d.min(), d.max())

rgba_img = cmap_custom(closest_indices.astype(int))
rgba_img[..., 3] = alpha

# ======================
# Energy color limits
# ======================
vmin = np.min(FH_total_energy_map)
vmax = np.max(FH_total_energy_map)

# ======================
# Higher-resolution figure
# ======================
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 6),     # increase physical size
    dpi=300,             # increase display resolution
    gridspec_kw={"width_ratios": [1, 1]}
)

ax0, ax1 = axes

# ----------------------
# Left: closest memory map
# ----------------------
ax0.imshow(
    rgba_img,
    interpolation="nearest"
)
ax0.set_xticks([])
ax0.set_yticks([])
ax0.invert_yaxis()
ax0.set_title("Closest Memory", fontsize=16)

# ----------------------
# Right: total energy map
# ----------------------
im = ax1.imshow(
    FH_total_energy_map,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest"
)
ax1.set_xticks([])
ax1.set_yticks([])
ax1.invert_yaxis()
ax1.set_title("Total Energy", fontsize=16)

cbar = fig.colorbar(
    im,
    ax=ax1,
    fraction=0.046,
    pad=0.04
)
cbar.set_label("Total Energy", fontsize=14)
cbar.ax.tick_params(labelsize=12)

# Make room on the far right for alpha mini legends
fig.subplots_adjust(right=0.84, wspace=0.15)

# Need finalized axis positions
fig.canvas.draw()
pos = ax0.get_position()

# ======================
# Vertical alpha legends
# ======================
legend_gap = 0.008
legend_width = 0.018
legend_x_offset = 0.015

total_height = pos.height
box_height = (total_height - (K - 1) * legend_gap) / K

n_alpha = 200

for k in range(K):

    bottom = pos.y0 + (K - 1 - k) * (box_height + legend_gap)

    ax_leg = fig.add_axes([
        pos.x1 + legend_x_offset,
        bottom,
        legend_width,
        box_height
    ])

    color = np.array(cmap_custom(k))

    bar = np.ones((n_alpha, 1, 4))
    bar[:, :, :3] = color[:3]
    bar[:, :, 3] = np.linspace(0.15, 1.0, n_alpha)[:, None]

    ax_leg.imshow(
        bar,
        origin="lower",
        aspect="auto",
        interpolation="nearest"
    )

    ax_leg.set_xticks([])
    ax_leg.set_yticks([])

    for spine in ax_leg.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

# Save high-resolution figure
plt.savefig(
    "closest_memory_total_energy_high_res.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Load saved frames

In [ ]:
# Download saved simulation frames from GitHub
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/YanchengDu/LiquidCHL/main/Spatial_Phase/plotted_figures.npy",
    "plotted_figures.npy",
)
print("Downloaded plotted_figures.npy")
frames_concat = np.load("plotted_figures.npy")


# Plot single memory

In [99]:
def plot_single_memory(memories):
    # reshape to 4x8
    memories_4x8 = memories.reshape(4, 8)

    vmin = 0.005#memories.min()
    vmax = 0.055#memories.max()

    fig, axes = plt.subplots(1,1,figsize=(15, 3))

    ax = axes

    ax.imshow(
        memories_4x8,
        cmap="gray_r",
        vmin=vmin+0.001,
        vmax=vmax,
        interpolation="none",
        aspect="equal"   # same geometry as binary plot
    )

    # draw grid lines
    ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
    ax.grid(which="minor", color="gray", linewidth=1)

    # remove tick labels
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_title(f"Memory", fontsize=16)

    plt.tight_layout()
    plt.show()

In [129]:
x,y = 70,105
plot_single_memory(frames_concat[-1,:,x,y])

# Plot saved snapshots

In [39]:
final_frame = frames_concat[-1]

n_show = min(nc, final_frame.shape[0])
ncols = 8
nrows = int(np.ceil(n_show / ncols))

vmin = np.min(final_frame[:n_show])
vmax = np.max(final_frame[:n_show])

TITLE_SIZE = 24
CBAR_LABEL_SIZE = 32
CBAR_TICK_SIZE = 20

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(4 * ncols, 3.5 * nrows)
)
axes = np.atleast_1d(axes).ravel()

for idx in range(n_show):
    im = axes[idx].imshow(
        final_frame[idx],
        origin="lower",
        vmin=vmin,
        vmax=vmax
    )
    axes[idx].text(
    0.03, 0.97,
    f"{idx+1}",
    transform=axes[idx].transAxes,
    fontsize=32,
    fontweight="bold",
    color="white",
    ha="left",
    va="top",
    bbox=dict(facecolor="black", alpha=0.5, pad=2)
)
    axes[idx].axis("off")

for ax in axes[n_show:]:
    ax.axis("off")

# Shared colorbar
cbar = fig.colorbar(
    im,
    ax=axes[:n_show],
    fraction=0.02,
    pad=0.02
)

cbar.set_label(
    "Volume Fraction",
    fontsize=CBAR_LABEL_SIZE
)

cbar.ax.tick_params(
    labelsize=CBAR_TICK_SIZE
)

fig.subplots_adjust(
    left=0.02,
    right=0.88,   # leave room for colorbar
    bottom=0.02,
    top=0.98,
    wspace=0.01,
    hspace=0.02
)

plt.show()

In [146]:
# ======================
# Settings
# ======================
nx = ny = 128
nc = 32

frame_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # <-- change this

K = 5
desired_indices = [0, 4, 8, 10, 6] # select color

# ======================
# Load trained chi
# ======================
chi_trained = np.load("chi_trained.npy")

kx = 2.0 * np.pi * np.fft.fftfreq(nx, d=1.0)
ky = 2.0 * np.pi * np.fft.fftfreq(ny, d=1.0)
K2 = kx[:, None] ** 2 + ky[None, :] ** 2

print("frames_concat shape:", frames_concat.shape)

# ======================
# Colormap for memory map
# ======================
base_colors = plt.cm.get_cmap("tab20")
custom_colors = [base_colors.colors[i] for i in desired_indices]
cmap_custom = ListedColormap(custom_colors)

# ======================
# Compute maps for selected frames
# ======================
rgba_imgs = []
energy_maps = []

for idx in frame_indices:
    frame = frames_concat[idx]

    FH_local_energy_map, FH_grad_energy_map, FH_total_energy_map = compute_energy_map_jax(
        frame,
        chi_trained,
        kappa=5.0 * jnp.ones(shape=nc),
        K2=K2
    )

    average_vectors, figure, closest_indices, min_distances = identify_memories(
        frame.transpose(1, 2, 0),
        partitioned_memories
    )

    # ----------------------
    # Convert distances to alpha
    # ----------------------
    d = min_distances.astype(float)

    d_min = 0.0
    d_max = 0.28

    d_norm = (d - d_min) / (d_max - d_min + 1e-12)
    d_norm = np.clip(d_norm, 0, 1)

    alpha = 1.0 - d_norm
    alpha = 0.05 + 0.95 * alpha

    # ----------------------
    # Construct RGBA image
    # ----------------------
    rgba_img = cmap_custom(closest_indices.astype(int))
    rgba_img[..., 3] = alpha

    rgba_imgs.append(rgba_img)
    energy_maps.append(np.array(FH_total_energy_map))

# ======================
# Shared energy color limits
# ======================
vmin = min(e.min() for e in energy_maps)
vmax = max(e.max() for e in energy_maps)

# ======================
# Plot: 2 rows x 10 frames + 1 colorbar column
# ======================
n_frames = len(frame_indices)

fig = plt.figure(figsize=(2.2 * n_frames, 4.8))

gs = gridspec.GridSpec(
    2,
    n_frames + 1,
    width_ratios=[1] * n_frames + [0.06],
    wspace=0.05,
    hspace=0.08
)

axes_top = []
axes_bottom = []

for i in range(n_frames):
    axes_top.append(fig.add_subplot(gs[0, i]))
    axes_bottom.append(fig.add_subplot(gs[1, i]))

# Colorbar axis on the very right, spanning both rows
cax = fig.add_subplot(gs[:, -1])

for i, idx in enumerate(frame_indices):

    # ----------------------
    # Top row: closest memory map
    # ----------------------
    ax_top = axes_top[i]

    ax_top.imshow(rgba_imgs[i], interpolation="nearest")
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.invert_yaxis()
    ax_top.set_title(f"Frame {idx}", fontsize=10)

    # ----------------------
    # Bottom row: total energy map
    # ----------------------
    ax_bottom = axes_bottom[i]

    im = ax_bottom.imshow(
        energy_maps[i],
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )

    ax_bottom.set_xticks([])
    ax_bottom.set_yticks([])
    ax_bottom.invert_yaxis()

# ======================
# Row labels
# ======================
axes_top[0].set_ylabel("Closest\nMemory", fontsize=12)
axes_bottom[0].set_ylabel("Total\nEnergy", fontsize=12)

# ======================
# Single shared colorbar on very right
# ======================
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Total Energy", fontsize=12)
cbar.ax.tick_params(labelsize=10)

plt.show()